In [5]:
import numpy as np

np.random.seed(85)

MIN_GREEN = 15.0
MAX_GREEN = 60.0
CYCLE_LENGTH = 120.0
NUM_PHASES = 4

TRAFFIC_VOLUMES = np.array([45.0, 30.0, 15.0, 25.0])

def calculate_intersection_delay(green_times):
    normalized_green = (green_times / np.sum(green_times)) * CYCLE_LENGTH
    delay_per_phase = (TRAFFIC_VOLUMES ** 2) / (normalized_green + 1e-5)
    return np.sum(delay_per_phase) * 10.0

def enforce_traffic_constraints(green_times):
    return np.clip(green_times, MIN_GREEN, MAX_GREEN)

population_size = 20
generations = 40
mutation_rate = 0.20
num_elites = 2

population = np.random.uniform(MIN_GREEN, MAX_GREEN, (population_size, NUM_PHASES))
for i in range(population_size):
    population[i] = enforce_traffic_constraints(population[i])

fitness_scores = np.array([calculate_intersection_delay(ind) for ind in population])
best_idx = np.argmin(fitness_scores)
initial_best_plan = population[best_idx].copy()
initial_best_delay = fitness_scores[best_idx]

for gen in range(generations):
    fitness_scores = np.array([calculate_intersection_delay(ind) for ind in population])

    sorted_indices = np.argsort(fitness_scores)
    population = population[sorted_indices]
    fitness_scores = fitness_scores[sorted_indices]

    new_population = np.zeros_like(population)
    new_population[:num_elites] = population[:num_elites]

    inv_fitness = 1.0 / (fitness_scores + 1e-5)
    selection_probs = inv_fitness / np.sum(inv_fitness)

    for i in range(num_elites, population_size):
        parent1_idx = np.random.choice(population_size, p=selection_probs)
        parent2_idx = np.random.choice(population_size, p=selection_probs)
        parent1 = population[parent1_idx]
        parent2 = population[parent2_idx]

        crossover_point = np.random.randint(1, NUM_PHASES)
        child = np.concatenate([parent1[:crossover_point], parent2[crossover_point:]])

        if np.random.rand() < mutation_rate:
            mutation_vector = np.random.uniform(-6, 6, NUM_PHASES)
            child = child + mutation_vector

        new_population[i] = enforce_traffic_constraints(child)

    population = new_population

final_fitness = np.array([calculate_intersection_delay(ind) for ind in population])
final_best_idx = np.argmin(final_fitness)
g_best = population[final_best_idx]
g_best_fitness = final_fitness[final_best_idx]

final_optimized_plan = (g_best / np.sum(g_best)) * CYCLE_LENGTH
initial_normalized_plan = (initial_best_plan / np.sum(initial_best_plan)) * CYCLE_LENGTH
improvement_percentage = ((initial_best_delay - g_best_fitness) / initial_best_delay) * 100

print("=" * 65)
print("TRAFFIC SIGNAL CONTROLLER: GENETIC ALGORITHM (GA) RUN")
print("=" * 65)
print(f"Traffic Volume Baseline (Phases 1-4): {TRAFFIC_VOLUMES} cars/min")
print("-" * 65)
print("INITIAL STATE (Random Parent Population Best):")
print(f" -> Assigned Phase Durations : {np.round(initial_normalized_plan, 2)} seconds")
print(f" -> Overall Intersection Delay: {initial_best_delay:.2f} cost units")
print("-" * 65)
print("FINAL STATE (Optimized Evolutionary Plan):")
print(f" -> Assigned Phase Durations : {np.round(final_optimized_plan, 2)} seconds")
print(f" -> Overall Intersection Delay: {g_best_fitness:.2f} cost units")
print("-" * 65)
print(f"EFFICIENCY GAIN: Delay reduced by {improvement_percentage:.2f}%!")
print("=" * 65)


TRAFFIC SIGNAL CONTROLLER: GENETIC ALGORITHM (GA) RUN
Traffic Volume Baseline (Phases 1-4): [45. 30. 15. 25.] cars/min
-----------------------------------------------------------------
INITIAL STATE (Random Parent Population Best):
 -> Assigned Phase Durations : [40.64 32.37 27.06 19.93] seconds
 -> Overall Intersection Delay: 1173.03 cost units
-----------------------------------------------------------------
FINAL STATE (Optimized Evolutionary Plan):
 -> Assigned Phase Durations : [47.07 31.08 15.39 26.47] seconds
 -> Overall Intersection Delay: 1102.19 cost units
-----------------------------------------------------------------
EFFICIENCY GAIN: Delay reduced by 6.04%!
